* https://github.com/hse-ling-python/seminars/blob/master/vector_models/vector_models_25_26.ipynb
* https://networkx.org/documentation/stable/reference/algorithms/community.html

Идея: берем модель Word2Vec, как вершины берем слова, соединяем ребрами слова, которые модель считает наиболее похожими (на данный момент просто 5 самых похожих), ищем сообщества специальным алгоритмом, получаем лексические кластеры  
Возможно стоило брать не 5 самых частотных а какой-то порог  
Идея не моя и возможно очевидная  
Хорошо бы еще добавить взвешивание

In [72]:
import networkx as nx
from gensim.models import KeyedVectors
import numpy as np
from tqdm import tqdm
import joblib

Качаем модель:

In [ ]:
!wget https://rusvectores.org/static/models/rusvectores4/RNC/ruscorpora_upos_skipgram_300_5_2018.vec.gz

Загружаем модель:

In [4]:
model = KeyedVectors.load_word2vec_format('ruscorpora_upos_skipgram_300_5_2018.vec.gz')

In [10]:
model.most_similar('кошка_NOUN', topn=5)

[('кот_NOUN', 0.7805721759796143),
 ('собака_NOUN', 0.7084056735038757),
 ('котенок_NOUN', 0.6833623647689819),
 ('мяукать_VERB', 0.661374568939209),
 ('мяукать_NOUN', 0.6479178071022034)]

Строим граф:

In [71]:
w2v_graph = nx.Graph()
for word in tqdm(model.index_to_key):
    w2v_graph.add_node(word)
    w2v_graph.add_edges_from(list(zip([word] * 5, [i[0] for i in model.most_similar(word, topn=5)])))

100%|██████████| 195071/195071 [36:18<00:00, 89.56it/s]  


In [73]:
joblib.dump(w2v_graph, 'w2v_graph.joblib')

['w2v_graph.joblib']

Создаем класс для хранения кластеров и графа:

In [ ]:
class SemClusters():
    def __init__(self):
        pass

    def create_from_graph(self, graph: nx.Graph):
        '''
        Create list of clusters from a given graph
        '''
        self.clusters = [i for i in nx.algorithms.community.label_propagation.asyn_lpa_communities(graph)]
        self.graph = graph

        return self

    def create_from_model(self, model):
        '''
        Create list of clusters from a given word2vec model
        '''
        w2v_graph = nx.Graph()
        for word in tqdm(model.index_to_key):
            w2v_graph.add_node(word)
            w2v_graph.add_edges_from(list(zip([word] * 5, [i[0] for i in model.most_similar(word, topn=5)])))

        return self.create_from_graph(self, w2v_graph)

    def get_cluster(self, word):
        '''
        Get cluster to which a word belongs
        '''
        res = []
        for cluster in self.clusters:
            if word in cluster:
                res.append(cluster)

        return res

    def save(self, name):
        '''
        Save model via joblib
        '''
        joblib.dump(self.clusters, name)

    def get_graph(self):
        '''
        Returns graph from which model was created
        '''
        return self.graph

    def get_clusts(self):
        '''
        Get list of all clusters
        '''
        return self.clusters

In [75]:
clusts = SemClusters().create_from_graph(w2v_graph)

In [88]:
joblib.dump(clusts, 'clusts.joblib')

['clusts.joblib']

Протестируем:

In [122]:
clusts.get_cluster('собирать_VERB')

[{'собираемый_VERB',
  'собирать_ADV',
  'собирать_NOUN',
  'собирать_VERB',
  'собрать_VERB'}]

Сколько получилось кластеров:

In [110]:
len(clusts.clusters)

10181

## Louvain

Сделаем другим алгоритмом поиска сообществ:

In [111]:
louvain = nx.algorithms.community.louvain.louvain_communities(w2v_graph)

In [112]:
len(louvain)

36

In [115]:
def get_cluster(clusters, word):
    '''
    Get cluster to which a word belongs
    '''
    res = []
    for cluster in clusters:
        if word in cluster:
            res.append(cluster)

    return res

In [117]:
get_cluster(louvain, 'стул_NOUN')

[{'горбинка_NOUN',
  'мастериец_NOUN',
  'гроссбух_NOUN',
  'подправить_VERB',
  'тщедушный_ADJ',
  'мочало_VERB',
  'несессер_NOUN',
  'высоконький_ADJ',
  'титька_NOUN',
  'колодкий_NOUN',
  'чистенько_ADV',
  'худущий_ADJ',
  'кудель_NOUN',
  'вышиваться_VERB',
  'обронить_ADJ',
  'облачять_VERB',
  'бриться_VERB',
  'кучерявый_ADJ',
  'скуфья_NOUN',
  'волосатый_ADJ',
  'перчатка_NOUN',
  'полушубочка_NOUN',
  'сшить_NOUN',
  'наручный_ADJ',
  'распашный_ADJ',
  'колгота_NOUN',
  'вызолотить_ADJ',
  'слабогрудый_ADJ',
  'поддевка_NOUN',
  'переметный_ADJ',
  'подбородочек_NOUN',
  'отцепляться_VERB',
  'шенкель_NOUN',
  'чубчик_NOUN',
  'ефрейторский_ADJ',
  'сосец_NOUN',
  'подделыватель_NOUN',
  'плетать_NOUN',
  'головка_NOUN',
  'чернобородый_ADJ',
  'люрекс_NOUN',
  'папиросный_ADJ',
  'гуччи_NOUN',
  'запыливать_VERB',
  'буфы_NOUN',
  'возиться_VERB',
  'анжела_NOUN',
  'прибирать_VERB',
  'натянутый_ADJ',
  'пинжак_NOUN',
  'сложенный_VERB',
  'четырехконечный_ADJ',
  'худо

## Другой label propagation

In [ ]:
def create_from_graph_lprop(graph: nx.Graph):
    '''
    Create list of clusters from a given graph
    '''
    return [i for i in nx.algorithms.community.label_propagation.label_propagation_communities(graph)]

In [125]:
lprop = create_from_graph_lprop(w2v_graph)

In [126]:
len(lprop)

8981

In [163]:
get_cluster(lprop, 'тяжелый_ADJ')

[{'-очедывать_ADV',
  '-очень_ADV',
  'веселотец_NOUN',
  'дружно_ADJ',
  'дружно_ADV',
  'дружный_ADJ',
  'забавно_ADJ',
  'завидно_ADJ',
  'заманчиво_ADJ',
  'занимательный_ADV',
  'занятно_ADV',
  'занятный_ADV',
  'захватывающий_ADV',
  'играючи_VERB',
  'легкотец_NOUN',
  'ловкий_ADV',
  'ловко_ADJ',
  'ловко_NOUN',
  'натурально_ADV',
  'непринужденно_ADJ',
  'обиднотец_NOUN',
  'очедывать_ADV',
  'очедывать_NOUN',
  'очень_NOUN',
  'поганый_VERB',
  'поучительно_ADV',
  'похоже_VERB',
  'приятно_NOUN',
  'приятнотец_NOUN',
  'скучно_ADJ',
  'скучно_ADV',
  'скучноотреть_NOUN',
  'скучный_ADV',
  'скучный_NOUN',
  'слаженно_ADV',
  'спориться_VERB',
  'споро_ADJ',
  'споро_ADV',
  'споро_NOUN',
  'тяжело_NOUN',
  'тяжелотец_NOUN',
  'тяжелый_ADJ',
  'тяжелый_NOUN',
  'тяжелый_VERB',
  'тяжко_NOUN',
  'уверенно_ADJ',
  'увлекательно_ADJ',
  'увлекательно_ADV',
  'удобнотец_NOUN',
  'ходко_ADV'}]

In [164]:
model.most_similar('тяжелый_ADJ')

[('тяжкий_ADJ', 0.7141361832618713),
 ('тяжелый_NOUN', 0.6850666403770447),
 ('тяжелый_ADV', 0.6037251353263855),
 ('тягостный_ADJ', 0.5810243487358093),
 ('тяжело_ADV', 0.5770506858825684),
 ('тяжелый_VERB', 0.5717610716819763),
 ('тяжко_ADV', 0.5629217028617859),
 ('тяжелейший_ADJ', 0.5485408306121826),
 ('тяжело_ADJ', 0.5484529137611389),
 ('легкий_ADJ', 0.5403433442115784)]